# Lab 03 — Resposta em frequência e identificação experimental

**Unidade I — Modelagem e análise de sistemas físicos** · conteúdo 1.2 do PPC

**Objetivos:**
1. Compreender experimentalmente o conceito de resposta em frequência;
2. Levantar o diagrama de Bode **ponto a ponto**, como em bancada;
3. Ajustar um modelo por assíntotas ao Bode experimental;
4. Validar o modelo no domínio do tempo.

**Referências:** Åström & Murray (FBS), cap. 9 · Ogata, cap. 7.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

rng = np.random.default_rng(7)

## 1. O experimento fundamental: senoide entra, senoide sai

Aplicando $u(t) = \sin(\omega t)$ a um sistema linear estável, a saída **em regime** é uma
senoide de mesma frequência com amplitude $|G(j\omega)|$ e defasagem $\angle G(j\omega)$.

In [ ]:
G = ct.tf([10], [1, 3, 10])   # planta desta aula: 2ª ordem
omega_test = 3.0              # frequência de teste [rad/s] (próxima da natural)

t = np.linspace(0, 20, 4000)
u = np.sin(omega_test * t)
resp = ct.forced_response(G, t, u)

# valores teóricos em omega_test
fr = ct.frequency_response(G, [omega_test])
mag_teo = np.abs(fr.frdata[0].item())
fase_teo = np.degrees(np.angle(fr.frdata[0].item()))

plt.figure(figsize=(10, 4))
plt.plot(t, u, alpha=0.7, label='entrada u(t)')
plt.plot(resp.time, resp.outputs, lw=2, label='saída y(t)')
plt.axhline(mag_teo, color='r', ls=':', label=f'|G(j{omega_test})| = {mag_teo:.2f}')
plt.axhline(-mag_teo, color='r', ls=':')
plt.xlabel('Tempo [s]')
plt.title(f'Regime senoidal: ganho {mag_teo:.2f}, fase {fase_teo:.1f}°')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

Após o transitório inicial (≈ 3 s), a saída oscila com amplitude e defasagem constantes:
esse **par (ganho, fase)** é um ponto do diagrama de Bode.

## 2. Levantamento do Bode ponto a ponto (procedimento de bancada)

Vamos automatizar o procedimento que em laboratório seria feito com gerador de sinais e
osciloscópio: para cada frequência, aplicar a senoide, esperar o regime, e medir ganho e fase.
Adicionamos ruído para simular a medição real.

In [ ]:
def medir_ponto_bode(sys, omega, sigma_ruido=0.02, n_per=20, pts_per=200):
    """Mede |G(jw)| e fase [graus] aplicando senoide e analisando o regime permanente."""
    T_per = 2 * np.pi / omega                       # período do sinal
    t = np.linspace(0, n_per * T_per, n_per * pts_per)
    u = np.sin(omega * t)
    y = ct.forced_response(sys, t, u).outputs
    y = y + rng.normal(0, sigma_ruido, size=y.shape)  # ruído de medição

    # descartar o transitório: usar apenas a segunda metade do registro
    sel = t >= t[-1] / 2
    ts, us, ys = t[sel], u[sel], y[sel]

    # correlação com seno e cosseno (detecção síncrona / lock-in)
    ref_sin, ref_cos = np.sin(omega * ts), np.cos(omega * ts)
    a = 2 * np.mean(ys * ref_sin)   # componente em fase
    b_q = 2 * np.mean(ys * ref_cos)  # componente em quadratura
    ganho = np.hypot(a, b_q)
    fase = np.degrees(np.arctan2(b_q, a))
    return ganho, fase

# frequências de ensaio (espaçadas em escala logarítmica, como em bancada)
omegas = np.logspace(-1, 1.5, 15)
medidas = np.array([medir_ponto_bode(G, w) for w in omegas])
ganhos_exp, fases_exp = medidas[:, 0], medidas[:, 1]

for w, g, f in zip(omegas, ganhos_exp, fases_exp):
    print(f"w = {w:7.3f} rad/s | ganho = {g:6.3f} ({20*np.log10(g):7.2f} dB) | fase = {f:8.2f} graus")

## 3. Bode experimental × Bode teórico

In [ ]:
omega_dense = np.logspace(-1, 1.5, 300)
fr_teo = ct.frequency_response(G, omega_dense)
mag_dB_teo = 20 * np.log10(fr_teo.magnitude.squeeze())
fase_teo_deg = np.degrees(fr_teo.phase.squeeze())

fig, axs = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axs[0].semilogx(omega_dense, mag_dB_teo, 'C0', lw=2, label='teórico')
axs[0].semilogx(omegas, 20 * np.log10(ganhos_exp), 'ro', label='experimental')
axs[0].set_ylabel('Módulo [dB]')
axs[0].legend()
axs[0].grid(True, which='both')

axs[1].semilogx(omega_dense, fase_teo_deg, 'C0', lw=2)
axs[1].semilogx(omegas, fases_exp, 'ro')
axs[1].set_ylabel('Fase [graus]')
axs[1].set_xlabel(r'$\omega$ [rad/s]')
axs[1].grid(True, which='both')
fig.suptitle('Diagrama de Bode: pontos medidos sobre a curva teórica')
plt.show()

## 4. Leitura de assinaturas no Bode (identificação por assíntotas)

Do gráfico experimental de um sistema **desconhecido**, extraímos a estrutura do modelo:

1. **Ganho em baixa frequência** → $20\log_{10}K$ ⟹ $K$;
2. **Inclinação final** → nº de polos menos zeros (−40 dB/déc ⟹ 2 polos em excesso);
3. **Pico de ressonância** $M_r$ em $\omega_r$ → par complexo com
   $M_r = \dfrac{1}{2\zeta\sqrt{1-\zeta^2}}$ e $\omega_r = \omega_n\sqrt{1-2\zeta^2}$;
4. **Fase em −90°** ocorre aproximadamente em $\omega_n$ (para 2ª ordem).

Vamos aplicar em nossos dados:

In [ ]:
# 1) ganho estático a partir do primeiro ponto medido
K_hat = ganhos_exp[0]

# 2) pico de ressonância nos dados
i_pico = np.argmax(ganhos_exp)
Mr = ganhos_exp[i_pico] / K_hat          # pico normalizado pelo ganho DC
w_r = omegas[i_pico]

# 3) inverter Mr para achar zeta (raiz da equação do pico)
zeta_hat = np.sqrt(0.5 * (1 - np.sqrt(1 - 1 / Mr**2))) if Mr > 1 else 0.7
wn_hat = w_r / np.sqrt(1 - 2 * zeta_hat**2) if Mr > 1 else w_r

print(f"Identificado: K = {K_hat:.2f}, zeta = {zeta_hat:.3f}, wn = {wn_hat:.2f} rad/s")
print("Valores reais: K = 1.0, zeta = 0.474, wn = 3.162 rad/s")

G_hat = ct.tf([K_hat * wn_hat**2], [1, 2 * zeta_hat * wn_hat, wn_hat**2])
print("Modelo identificado:", G_hat)

## 5. Validação cruzada no domínio do tempo

Modelo identificado **em frequência** deve prever bem a resposta **ao degrau** — validação
em domínio diferente do usado no ajuste é a validação mais convincente.

In [ ]:
t_val = np.linspace(0, 6, 600)
r_real = ct.step_response(G, t_val)
r_hat = ct.step_response(G_hat, t_val)

plt.figure(figsize=(8, 4))
plt.plot(r_real.time, r_real.outputs, lw=2, label='planta real')
plt.plot(r_hat.time, r_hat.outputs, '--', lw=2, label='modelo identificado (via Bode)')
plt.xlabel('Tempo [s]')
plt.ylabel('y(t)')
plt.title('Validação no tempo do modelo identificado em frequência')
plt.legend()
plt.grid(True)
plt.show()

---
> **🖼️ Figuras de apoio nos livros:**
> - Schaum (DiStefano), **Fig. 15-3** — exemplo de diagrama de Bode (módulo e fase). Cap. 15, **p. 366** (p. 377 do PDF).

## Exercícios (relatório do Lab 03)

**E1.** Repita o levantamento ponto a ponto para o circuito RC do Lab 01
($\tau = 1$ s) e determine graficamente a **frequência de canto** (queda de 3 dB).
Compare com $1/\tau$.

**E2.** Meça a fase da planta $G$ desta aula em $\omega = 10$ rad/s e explique por que ela se
aproxima de −180°.

**E3.** Uma planta desconhecida `G_mist = ct.tf([8], [1, 4.4, 8.4, 4])` deve ser identificada
pelo procedimento da Seção 2 (finja não conhecer os coeficientes). Levante o Bode, proponha a
estrutura (quantos polos?) e ajuste um modelo aproximado. Valide ao degrau.

**E4.** Adicione um tempo morto de 0,5 s à planta ($G \cdot e^{-0{,}5s}$, use `ct.pade`) e refaça
o Bode. O que muda no módulo? E na fase? Por que tempo morto é perigoso para controle?

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui